# 28 — Union-Find (Disjoint Set Union) (Amazon FAR style)

Track connectivity among `n` elements under merges — the backbone of Kruskal's MST, connected
components, and equivalence queries. Parts 1-3 build the core (concurrency at Part 3); Parts 4-5 are
stretch tasks (cycle/redundant-edge detection, then parallel connectivity). Fill stubs, run each test
cell, peek at the collapsed `ref_` solutions only after trying.

---

## Part 1 — Union-Find

`UnionFind(n)` over elements `0..n-1`: `find(x)` (with **path compression**), `union(x, y) -> bool`
(with **union by rank**; returns False if already in the same set), `connected(x, y) -> bool`.

**Lock down:** path compression + union by rank give near-O(1) amortized; `union` of already-connected
elements is a no-op returning False (this is the cycle signal in Part 4).

In [ ]:
class UnionFind:
    def __init__(self, n):
        raise NotImplementedError

    def find(self, x):
        raise NotImplementedError

    def union(self, x, y):
        raise NotImplementedError

    def connected(self, x, y):
        raise NotImplementedError

In [ ]:
# --- Part 1 tests ---
def _t1():
    uf = UnionFind(6)
    assert not uf.connected(0, 1)
    uf.union(0, 1); uf.union(1, 2)
    assert uf.connected(0, 2) and not uf.connected(0, 3)
    assert uf.union(0, 2) is False                     # already connected
    print("Part 1 OK")

_t1()

## Part 2 — Components

`num_components(uf, n) -> int` (number of distinct sets) and `component_sizes(uf, n) -> list`
(set sizes, sorted descending).

**Lock down:** group elements by their root (`find`); the number of distinct roots is the component
count.

In [ ]:
def num_components(uf, n):
    raise NotImplementedError


def component_sizes(uf, n):
    raise NotImplementedError

In [ ]:
# --- Part 2 tests ---
def _t2():
    uf = UnionFind(6)
    uf.union(0, 1); uf.union(2, 3); uf.union(3, 4)
    assert num_components(uf, 6) == 3                   # {0,1}, {2,3,4}, {5}
    assert component_sizes(uf, 6) == [3, 2, 1]
    print("Part 2 OK")

_t2()

## Part 3 — Thread-safe DSU

`ThreadSafeUnionFind(n)`: `find`/`union`/`connected` safe under concurrent unions. Many threads union
edges of one big chain at once; the whole chain ends up connected.

**The invariant:** unioning every edge of a 0—1—…—(n-1) chain (edges shuffled across 8 threads) leaves
a single component regardless of order. **Discuss:** `union` is a read-modify-write on shared
parent/rank arrays (guard it); since a locked `union` internally calls a locked `find`, you need a
**re-entrant** lock (`RLock`) or an unlocked internal helper — a plain `Lock` self-deadlocks. Finer-
grained locking is hard, which is why production DSU is usually single-writer or uses lock-free CAS.

In [ ]:
import threading


class ThreadSafeUnionFind:
    def __init__(self, n):
        raise NotImplementedError

    def find(self, x):
        raise NotImplementedError

    def union(self, x, y):
        raise NotImplementedError

    def connected(self, x, y):
        raise NotImplementedError

In [ ]:
# --- Part 3 tests ---
def _t3():
    import random
    n = 1000
    uf = ThreadSafeUnionFind(n)
    edges = [(i, i + 1) for i in range(n - 1)]
    random.Random(0).shuffle(edges)
    chunks = [edges[i::8] for i in range(8)]

    def worker(es):
        for a, b in es:
            uf.union(a, b)

    ts = [threading.Thread(target=worker, args=(c,)) for c in chunks]
    for t in ts: t.start()
    for t in ts: t.join()
    assert uf.connected(0, n - 1)
    assert len({uf.find(i) for i in range(n)}) == 1    # one component
    print("Part 3 OK")

_t3()

## Part 4 (stretch) — Build from edges, detect redundant edges

`build(n, edges) -> (uf, redundant)`: process an edge list; an edge whose endpoints are **already**
connected is *redundant* (it closes a cycle) — collect those. Validate node indices (out of range
raises `ValueError`).

**Discuss:** redundant edges are exactly the ones Kruskal's MST skips; cycle detection falls out of
`union` returning False; self-loops are redundant too.

In [ ]:
def build(n, edges):
    raise NotImplementedError

In [ ]:
# --- Part 4 tests ---
def _t4():
    uf, redundant = build(5, [(0, 1), (1, 2), (0, 2), (3, 4)])    # (0,2) closes a cycle
    assert redundant == [(0, 2)]
    assert num_components(uf, 5) == 2                  # {0,1,2}, {3,4}
    try:
        build(3, [(0, 5)])
    except ValueError:
        pass
    else:
        raise AssertionError("expected ValueError for out-of-range node")
    print("Part 4 OK")

_t4()

## Part 5 (stretch) — Parallel connectivity

`parallel_components(n, edges, num_procs) -> int`: split the edge list across processes; each worker
reduces its chunk to a **spanning forest** (only the edges that merged something) with
`dsu_workers.spanning_forest`; the parent unions those forests into one DSU and counts components.

**Discuss:** each chunk independently shrinks to ≤ n-1 edges (the parallel part); merging the forests
is a cheap sequential reduce; final components are the same as processing all edges serially.

In [ ]:
from concurrent.futures import ProcessPoolExecutor
import dsu_workers


def parallel_components(n, edges, num_procs):
    raise NotImplementedError

In [ ]:
# --- Part 5 tests ---
def _t5():
    n = 10
    edges = [(0, 1), (1, 2), (2, 0), (3, 4), (4, 5), (6, 7)]   # {0,1,2},{3,4,5},{6,7},{8},{9}
    assert parallel_components(n, edges, 3) == 5
    print("Part 5 OK")

_t5()

## Likely follow-ups
- Kruskal's MST (sort edges, union, skip redundant); number of connected components online.
- Union-find with rollback / persistence; weighted union for offline dynamic connectivity.
- Lock-free DSU (CAS on parent); why concurrent path compression is subtle.

---
## Reference solutions (don't peek until you've tried)

In [ ]:
import threading
from collections import Counter
from concurrent.futures import ProcessPoolExecutor
import dsu_workers


class RefUnionFind:
    def __init__(self, n):
        self.parent = list(range(n))
        self.rank = [0] * n

    def find(self, x):
        while self.parent[x] != x:
            self.parent[x] = self.parent[self.parent[x]]   # path halving
            x = self.parent[x]
        return x

    def union(self, x, y):
        rx, ry = self.find(x), self.find(y)
        if rx == ry:
            return False
        if self.rank[rx] < self.rank[ry]:
            rx, ry = ry, rx
        self.parent[ry] = rx
        if self.rank[rx] == self.rank[ry]:
            self.rank[rx] += 1
        return True

    def connected(self, x, y):
        return self.find(x) == self.find(y)


def ref_num_components(uf, n):
    return len({uf.find(i) for i in range(n)})


def ref_component_sizes(uf, n):
    c = Counter(uf.find(i) for i in range(n))
    return sorted(c.values(), reverse=True)


class RefThreadSafeUnionFind(RefUnionFind):
    def __init__(self, n):
        super().__init__(n)
        self.lock = threading.RLock()                  # re-entrant: union() calls find() under the lock

    def find(self, x):
        with self.lock:
            return super().find(x)

    def union(self, x, y):
        with self.lock:
            return super().union(x, y)

    def connected(self, x, y):
        with self.lock:
            return super().find(x) == super().find(y)


def ref_build(n, edges):
    uf = RefUnionFind(n)
    redundant = []
    for a, b in edges:
        if not (0 <= a < n and 0 <= b < n):
            raise ValueError("node out of range")
        if not uf.union(a, b):
            redundant.append((a, b))
    return uf, redundant


def ref_parallel_components(n, edges, num_procs):
    chunks = [edges[i::num_procs] for i in range(num_procs)]
    uf = RefUnionFind(n)
    with ProcessPoolExecutor(max_workers=num_procs) as ex:
        for keep in ex.map(dsu_workers.spanning_forest, [(n, c) for c in chunks]):
            for a, b in keep:
                uf.union(a, b)
    return ref_num_components(uf, n)


_u = RefUnionFind(4); _u.union(0, 1); _u.union(2, 3)
assert ref_num_components(_u, 4) == 2 and ref_component_sizes(_u, 4) == [2, 2]
_uf, _r = ref_build(4, [(0, 1), (1, 0), (2, 3)]); assert _r == [(1, 0)]
assert ref_parallel_components(6, [(0, 1), (1, 2), (3, 4)], 2) == 3
print("reference solutions OK")